# MV-Kubric WebDataset pilot

This notebook is a thin Modal Notebook driver. Attach the existing `jeet-mvtracker-data-v2` Volume at `/mnt/mvtracker-data`, select one T4 with 16 CPUs and 32 GB RAM, and run the CPU conversion function below. The function is tagged `owner=jeet`, `project=mvtracker`, `purpose=profiling` and logs progress to W&B.

The pilot converts a small explicit scene list into four-scene uncompressed TAR shards with NVIDIA `wds2idx` indices. The real MV-Kubric scenes carry ten source views; training can still select up to six. It does not delete native data or launch training.

In [ ]:
from pathlib import Path
import json

DATA_ROOT = Path('/mnt/mvtracker-data')
SCENE_ROOT = DATA_ROOT / 'datasets/kubric-multiview/train'
OUTPUT_ROOT = DATA_ROOT / 'datasets/kubric-multiview-webdataset/v1/train'
pilot_scenes = tuple(str(scene) for scene in range(1001, 1033))
print(f'{len(pilot_scenes)} scenes: {pilot_scenes[0]}..{pilot_scenes[-1]}')

## Convert the pilot

The `%modal` invocation is intentionally explicit. Use `shard_workers=8` only when eight CPU slots are available; otherwise leave it at one.

In [ ]:
# In Modal Notebook, invoke the deployed function with the repository checkout.
# %modal run tools/modal_mvkubric_webdataset.py::convert \
#   --scene-root /mnt/mvtracker-data/datasets/kubric-multiview/train \
#   --output-root /mnt/mvtracker-data/datasets/kubric-multiview-webdataset/v1/train \
#   --scenes 1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032 \
#   --scenes-per-shard 4 --shard-workers 1 --read-workers 16
print('Run the commented %modal command after attaching the Volume and selecting a T4.')

In [ ]:
manifest = OUTPUT_ROOT / 'manifest.json'
if manifest.exists():
    report = json.loads(manifest.read_text())
    print(f"converted scenes={len(report['scene_ids'])} shards={len(report['shards'])}")
    for shard in report['shards']:
        print(shard['name'], shard['scene_ids'], shard['status'])
else:
    print('No manifest yet; run the conversion cell first.')

## Native versus DALI benchmark

This measures the same 32 scenes at 1, 2, 4 and 6 selected views. It reports cold startup, warm read/unpack, GPU decode, exposed wait, samples/s, encoded bytes/s, and hardware samples. It does not run a model or optimizer.

In [ ]:
# In Modal Notebook, invoke the tagged one-T4 function:
# %modal run tools/modal_mvkubric_webdataset.py::benchmark \
#   --run-name mvkubric-webdataset-t4-pilot \
#   --scenes 1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032 \
#   --warmup 4 --measured 16 --workers 8
print('The benchmark cell is intentionally not launched automatically.')